#Use an AI-based tool to generate simple video animations.

In [1]:
!pip install -U diffusers

In [2]:
!pip install -U diffusers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 52.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video

# 2. Check if GPU is actually available
if not torch.cuda.is_available():
    print("⚠️ WARNING: You are using CPU. Please go to Runtime > Change runtime type > T4 GPU")
else:
    print(f"✅ Success: Using GPU ({torch.cuda.get_device_name(0)})")

# 3. Load the Model
# We strictly use float16 to fit in Colab's memory and avoid the mismatch error
pipe = DiffusionPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b",
    torch_dtype=torch.float16,
    variant="fp16"
)

# 4. optimize the scheduler
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# 5. Move to GPU (This fixes the 'bias type' error)
pipe.enable_model_cpu_offload()
# OR use pipe.to("cuda") if you have plenty of VRAM,
# but enable_model_cpu_offload() is safer on Colab free tier.

# 6. Generate
prompt = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"
print("Generating video...")

video_frames = pipe(prompt, num_frames=16).frames[0]

# 7. Save and Download
video_path = "astronaut.mp4"
export_to_video(video_frames, video_path)
print(f"Video saved to {video_path}")

# Optional: Display inside Colab
from IPython.display import Video
Video(video_path)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


✅ Success: Using GPU (Tesla T4)


model_index.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--damo-vilab--text-to-video-ms-1.7b/snapshots/8227dddca75a8561bf858d604cc5dae52b954d01/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


Generating video...


  0%|          | 0/50 [00:00<?, ?it/s]

Video saved to astronaut.mp4
